# DQ STG MSISDN IMSI OPERATOR

Визуализация последнего прогона `dq-stg-msisdn-imsi-operator` (логи `DQ_STG_MSISDN_IMSI_OPERATOR`) и профиль месячного parquet `stg_msisdn_imsi`.

Перед запуском: `uv run mobile build-stg-msisdn-imsi-operator`, затем `uv run mobile dq-stg-msisdn-imsi-operator`.

In [ ]:
from __future__ import annotations

import pandas as pd
from IPython.display import display

from mobile.project_paths import PROJECT_ROOT

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 120)

ROOT = PROJECT_ROOT
TAG = "DQ_STG_MSISDN_IMSI_OPERATOR"
BOUNDARY = "dataset_basic"

In [ ]:
from mobile.pipelines.nb.common import checks_by_status, load_dq_dashboard

try:
    dq_logs, latest, meta = load_dq_dashboard(ROOT, tag=TAG, boundary_check=BOUNDARY)
except (FileNotFoundError, ValueError) as exc:
    print(f"Нет DQ-логов для {TAG}: {exc}")
    latest = meta = None
else:
    print("Последний прогон:", meta)
    display(checks_by_status(latest))

## Метрики DQ (последний прогон)

In [ ]:
import matplotlib.pyplot as plt

from mobile.pipelines.nb.common import render_stg_msisdn_imsi_operator_dq_overview

if latest is None:
    print("Нет данных для визуализации")
else:
    fig = render_stg_msisdn_imsi_operator_dq_overview(latest)
    plt.show()

In [ ]:
from mobile.pipelines.nb.common import failed_warning_table, metrics_wide_table

if latest is None:
    pass
else:
    display(pd.DataFrame([meta]))
    print("\n--- failed / warning ---")
    display(failed_warning_table(latest))
    print("\n--- все метрики (плоская таблица) ---")
    wide = metrics_wide_table(latest)
    display(wide.head(80))
    if len(wide) > 80:
        print(f"... всего строк metrics: {len(wide)}")

## Профиль parquet `stg_msisdn_imsi`

Месяц берётся из последнего DQ-прогона (`dataset_basic.report_date`) или из самого свежего файла в `data/stg/msisdn_imsi/`.

In [ ]:
import matplotlib.pyplot as plt

from mobile.pipelines.nb.common import (
    _stg_binding_report_month,
    display_stg_msisdn_imsi_parquet_summary,
    render_stg_msisdn_imsi_parquet_profile,
)

report_month = _stg_binding_report_month(latest)
print(f"report_month: {report_month.isoformat()}")

try:
    display_stg_msisdn_imsi_parquet_summary(ROOT, report_month)
except FileNotFoundError as exc:
    print(exc)
else:
    fig = render_stg_msisdn_imsi_parquet_profile(ROOT, report_month)
    plt.show()